# DeepSpot-M: one slide to one result file

**This notebook takes an H&E image to predicted spatial gene expression, twice
over, because there are two routes and you have to pick one.**

| Route | Where DeepSpot-H runs | What leaves your machine | Observation kind |
|---|---|---|---|
| **Send your slides** | On ours | The tile images | `patches` |
| **Your slides stay with you** | On your machine | Numbers and positions. No image. | `embeddings` |

DeepSpot-H turns each tile of your slide into one embedding. DeepSpot-M turns
those embeddings into spatial gene expression, and it runs on the service on
both routes. The route decides only where the first step happens and therefore
what you upload.

Both routes end at the same file: one `.h5ad` with a row per tile, a column
per gene, and the tile's centre under `obsm["spatial"]`.

**What you need.** Python 3.10 or newer. The example costs about 220 MiB over
the network and about 600 MiB on disk once unpacked. A GPU is optional, and
only for the second route. Everything up to the submission runs without an
account; the submission needs a key, and
[Get access](https://docs.auroraomics.org/get-access/) is how you get one.

A prediction is a hypothesis about what an assay would have measured. It is
not a measurement and it has no clinical use.
[Responsible use](https://docs.auroraomics.org/responsible-use/) states the
conditions on every result.

## 0. Setup

Everything imported here comes from `requirements.txt`. The local route needs
one extra, `auroraomics[embed]`, and section 7 says where.

In [ ]:
import sys
from pathlib import Path

import h5py
import numpy as np

import auroraomics as ao
from auroraomics import contracts
from auroraomics.client import Client

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from quickstart import zenodo  # noqa: E402

print("auroraomics", ao.__version__)
print("python     ", sys.version.split()[0])

## 1. Configuration

**Set the route here.** It is the only choice in this notebook, and
everything downstream follows from it.

In [ ]:
from enum import Enum


class Route(str, Enum):
    """Where DeepSpot-H runs, and therefore what you upload."""

    SEND_SLIDES = "patches"     # tile images leave your machine
    KEEP_SLIDES = "embeddings"  # only numbers and positions leave


ROUTE = Route.SEND_SLIDES

# Put the tiles where the assay measured, so that every predicted spot has a
# measured counterpart to look at in notebook 02. Set this False and the
# slide is tiled regularly instead, which is what you would do for a slide
# that has no assay -- which is the case this service exists for.
USE_MEASURED_SPOT_CENTRES = True

DATA = ROOT / "data"
SAMPLE_ARCHIVE = DATA / "sample.zip"    # route: send your slides
SAMPLE_NPZ = DATA / "sample.npz"        # route: your slides stay with you
PREDICTION = DATA / "prediction.h5ad"

print("route:", ROUTE.name, "-- submitting", ROUTE.value)

## 2. The example slide, without the 1.9 GiB

The example is one lung-cancer section from an open dataset of Visium
sections with tertiary lymphoid structures. The record publishes it as a
single archive of about 1.9 GiB, and **this notebook does not download that
archive.**

A zip file keeps an index of its members at the end, and HTTP can ask for a
byte range. So `quickstart/zenodo.py` reads the index over two small requests,
then streams only the member you asked for and inflates it on the way to disk,
checking the member's own stored checksum as the bytes arrive. The slide costs
about 200 MiB on the wire; the measured expression about 18 MiB.

The same trick works on any published zip whose host answers a range request,
which is most of them. It is worth knowing.

In [ ]:
archive = zenodo.index()
print(f"{archive.filename}: {zenodo.human(archive.size)}, "
      f"{len(archive.members)} members\n")

for role in ("slide", "measured"):
    member = archive.matching(zenodo.role_pattern(role))
    print(f"{role:>9}: {member.name}")
    print(f"{'':>9}  {zenodo.human(member.compressed_size)} on the wire, "
          f"{zenodo.human(member.size)} on disk")

In [ ]:
slide_path = zenodo.fetch("slide", DATA, on_progress=zenodo.progress)
measured_path = zenodo.fetch("measured", DATA, on_progress=zenodo.progress)
print(slide_path)
print(measured_path)

> Dawo, S., Nonchev, K., & Silina, K. (2025). *10x Visium Spatial
> Transcriptomics Dataset: Kidney (3) and Lung (5) Cancer with Tertiary
> Lymphoid Structures* [Data set]. Zenodo.
> <https://doi.org/10.5281/zenodo.14620362>
>
> Licensed CC-BY-4.0. If you re-use this data, cite it.

## 3. Open the slide, and read its scale from the slide

A tile is a **physical** size, not a pixel count: the model reads a square of
tissue a fixed number of micrometres across, and how many pixels that is
depends on how finely your slide was scanned. So the scale has to be right.

`open_slide` reads it out of the file's own metadata and keeps it as
`recorded_mpp`. You may override it, and an override wins, because a converted
file can carry a resolution its converter invented. **Do not guess one.** A
tile declared at the wrong scale gives a wrong prediction and nothing raises an
error.

`open_slide` reads only the square being cut, so the slide never has to fit in
memory.

In [ ]:
slide = ao.open_slide(slide_path)

height, width, _ = slide.shape
print(f"{width:,} x {height:,} px")
print(f"recorded_mpp: {slide.recorded_mpp} um/px  (from the file's own metadata)")
print(f"in use:       {slide.mpp} um/px")
print(f"that is {width * slide.mpp / 1000:.1f} x {height * slide.mpp / 1000:.1f} mm of glass")

### Check the scale against something that does not come from the file

The file says what it says. Here there is a second, independent measurement
available: this section was run on an assay whose capture spots sit on a grid
with a known physical spacing, and the dataset records where those spots are
in slide pixels. Measure the spacing in pixels, divide, and you have the scale
the assay implies.

Two numbers from two different places agreeing to a fraction of a percent is
worth thirty seconds. If your own slide has no second source, read the scanner
metadata and trust it, or measure something of a known size.

In [ ]:
with h5py.File(measured_path) as handle:
    # This dataset's own columns. `x_pixel` is the ROW and `y_pixel` is the
    # COLUMN in full-resolution slide pixels -- verified by drawing the spots
    # over the section's own image, which is in the same file.
    spot_row = handle["obs/x_pixel"][:]
    spot_col = handle["obs/y_pixel"][:]

SPOT_PITCH_UM = 100.0  # centre-to-centre spacing of this assay's capture grid

points = np.stack([spot_col, spot_row], axis=1)
sample = np.random.default_rng(0).choice(len(points), size=800, replace=False)
gaps = np.linalg.norm(points[sample, None, :] - points[None, :, :], axis=-1)
gaps[gaps < 1e-9] = np.inf
pitch_px = float(np.median(gaps.min(axis=1)))

implied_mpp = SPOT_PITCH_UM / pitch_px
disagreement = abs(implied_mpp - slide.mpp) / slide.mpp

print(f"median spot spacing: {pitch_px:.2f} px")
print(f"implied by the assay: {implied_mpp:.6f} um/px")
print(f"recorded by the file: {slide.mpp:.6f} um/px")
print(f"they disagree by {disagreement:.2%}")

## 4. Where the tiles go

One tile, one spot: a row of the result is the prediction at the centre of one
tile, and `obsm["spatial"]` carries that centre back in full-resolution slide
pixels.

By default the slide is tiled regularly with no overlap, which is what you
want for a slide with no assay behind it. Here there is one, so the tiles are
put at the measured spot centres. That is what makes the side-by-side in
notebook 02 a comparison of the same places rather than of two different
griddings of the same tissue.

Centres are `(column, row)`.

In [ ]:
if USE_MEASURED_SPOT_CENTRES:
    centres = [(int(c), int(r)) for c, r in zip(spot_col, spot_row)]
    print(f"{len(centres):,} tile centres, taken from the measured spots")
    print("first three:", centres[:3])
else:
    centres = None
    print("centres = None -- the slide is tiled regularly with no overlap")

## 5. What the service requires

Three values are the service's, not yours, and reading them rather than
copying them is what keeps your machine and the service in agreement:

- **the quality floors** every tile is judged against, which come off
  discovery and **need no key**;
- **the tile's physical size**, which is a property of the model and is stated
  on its card;
- **the model id** you submit against.

The catalogue needs a key with the `read` scope. This is the first cell in the
notebook that does.

In [ ]:
# The quota counters, read from the contract document shipped inside the
# package. No network, no key: this is what the service enforces, not a
# copy of it.
counters = contracts.load("public-api-counters.tokens.json")
tier = counters["defaultTier"]
print(f"default tier: {tier}")
for counter, limit in sorted(counters["tiers"][tier].items()):
    window = counters["counters"][counter]["window"]
    print(f"  {counter:<12} {limit:>9,} per {window}")

In [ ]:
from dataclasses import asdict

client = Client()
print("api:", client.base_url)
print("key:", client.key_prefix or "none stored -- run `auroraomics login you@institution.edu`")

# Discovery, and therefore the floors, answer without a credential.
floors = client.qc_thresholds()
print("\nquality floors, from the service:")
for name, value in asdict(floors).items():
    print(f"  {name:<24} {value}")

In [ ]:
# The catalogue. This call needs the key.
cards = client.models()
card = next(c for c in cards if ROUTE.value in c["accepts"]["observations"])

patch_um = float(card["input_spec"]["patch_um"])

# The package works this out for itself from `patch_um` and `mpp`; it is
# computed here only so you can see what the physical size becomes on YOUR
# slide. A fractional pixel count is truncated.
tile_px = int(patch_um / slide.mpp)

print("model:      ", card["id"])
print("reads:      ", card["accepts"]["observations"], "+", card["accepts"]["covariates"])
print("returns:    ", f"{card['output']['n_genes']:,} genes")
print("value scale:", card["output"]["transform"]["name"])
print()
print(f"tile edge:   {patch_um:g} um, which on this slide is {tile_px} px")

`transform` above names the scale the returned values are on. Note it now:
notebook 02 draws these values, and a colour scale laid over values whose
scale you have not looked at is a picture of an assumption.

A covariate is a fact the image does not carry. This model also accepts a bulk
RNA profile, which costs nothing against your quota and which
[Add a bulk RNA profile](https://docs.auroraomics.org/guides/bulk-rna/)
explains. This notebook sends none.

## 6. Route: send your slides

`pack_tiles` cuts the tiles, judges each one, writes the ones that pass into a
single archive, and reports what it dropped and why. Tiles stream past one at
a time, so a whole slide packs on a laptop.

In [ ]:
if ROUTE is Route.SEND_SLIDES:
    report = ao.pack_tiles(
        slide,
        SAMPLE_ARCHIVE,
        mpp=slide.mpp,          # read from the slide in section 3
        patch_um=patch_um,      # read from the model's card in section 5
        centers=centres,
        thresholds=floors,      # read from the service in section 5
    )

    print(f"{report.written:,} tiles kept of {report.considered:,} considered")
    print(f"{report.rejected:,} dropped by quality control:")
    for reason, count in sorted(report.rejected_by_reason.items()):
        print(f"  {reason:<24} {count:,}")
    if report.out_of_bounds:
        print(f"{report.out_of_bounds:,} centres whose square ran off the slide")

### Read the drop count before you go on

A drop count is information, not a failure. Each reason names the floor the
tile did not reach:

- **`foreground_ratio`** — too little of the square was tissue. At the rim of
  a capture area a spot's square is mostly slide glass, and a square of glass
  carries nothing for a model to read.
- **`hsv_tissue_ratio`** — too little of it read as stained tissue.
- **`blur_laplacian_var`** — too little detail; the square is out of focus.

In a trial pack of the first 400 spot centres on this section, 370 squares
were kept and every one of the 30 losses was `foreground_ratio`, at the rim of
the capture area. That is the shape to expect here. A run where most tiles go
for `blur_laplacian_var`, or where the losses sit in the middle of the tissue
rather than at its rim, is telling you something about the scan rather than
about the tissue.
[Check tile quality](https://docs.auroraomics.org/check-tiles/) says what to
do about each.

In [ ]:
if ROUTE is Route.SEND_SLIDES:
    # Validate the archive before uploading it. `read_archive` checks the
    # member names, the manifest and the geometry before it decodes a pixel.
    with ao.read_archive(SAMPLE_ARCHIVE) as arc:
        first = next(arc.tiles())
        print(f"{len(arc):,} tiles at {arc.mpp:.6f} um/px")
        print(f"first tile: {first.image.shape} with its top-left at {(first.x, first.y)}")
        print(f"thumbnail: {arc.has_thumbnail}   provenance: {arc.has_provenance}")

    print(f"\narchive on disk: {zenodo.human(SAMPLE_ARCHIVE.stat().st_size)}")

The archive holds the tiles, a manifest giving each one's position and its
quality measurements, and optionally a small overview image for a viewer to
draw on. It holds no slide label and no macro image, and the package never
produces them. On this route those tile images are what leaves your machine.

## 7. Route: your slides stay with you

On this route you run DeepSpot-H yourself and **no image is uploaded**. What
leaves is a row of numbers per tile, the tile's centre and the quality
measurements that kept it.

It needs one extra, which carries an encoder runtime but not the weights:

```
pip install "auroraomics[embed]"
```

The weights are fetched once from wherever the service pins them and cached.
Reading that pin is what your key is for here: it fixes which weights are
meant, so two machines compute the same embedding. It runs on a CPU; a GPU is
used when there is one.

`embed_tiles` takes any sliceable `(height, width, 3)` RGB array, and the
open slide from section 3 is one: reading `slide[y0:y1, x0:x1]` pulls only the
part of the file that window touches. So a slide larger than memory works, and
you pass the same object on both routes.

In [ ]:
if ROUTE is Route.KEEP_SLIDES:
    # `slide` is read as slide[y0:y1, x0:x1] and only the strips a window
    # touches come off the disk, so it goes straight in: one square exists at
    # a time, whatever the slide's size.
    encoder = ao.resolve_encoder(client.encoders())
    embed_report = ao.embed_tiles(
        slide,
        SAMPLE_NPZ,
        mpp=slide.mpp,
        patch_um=patch_um,
        encoder=encoder,
        thresholds=floors,
        centers=centres,
    )

    print(f"{embed_report.rows:,} rows of {embed_report.dim} numbers")
    print(f"cut at {embed_report.patch_px} px, from {embed_report.considered:,} centres")
    for reason, count in sorted(embed_report.rejected_by_reason.items()):
        print(f"  {reason:<24} {count:,}")

### Open the file and see exactly what would leave

This is the cell to run in front of a data-governance review. The file is a
plain `.npz`; everything in it is listed below, and none of it is pixels.

In [ ]:
if ROUTE is Route.KEEP_SLIDES:
    with np.load(SAMPLE_NPZ, allow_pickle=False) as bundle:
        for key in bundle.files:
            value = bundle[key]
            print(f"  {key:<20} {str(value.shape):<14} {value.dtype}")

    print(f"\nfile on disk: {zenodo.human(SAMPLE_NPZ.stat().st_size)}")
    print("no tile, no thumbnail, no slide label, no macro image, no scanner metadata")

These numbers are derived from the image, and partial reconstruction of image
content from such numbers is a known research capability. Aurora attempts none
and has no path to your pixels.
[Privacy and retention](https://docs.auroraomics.org/privacy/) is the full
statement.

## 8. Submit, and wait

The submission is the same call on both routes, with one word different: the
observation kind. A path is uploaded first, in parts, so an interrupted
upload resumes rather than restarting.

One job per key runs at a time. The job carries its own queue position and an
estimated wait, and `wait` polls at the interval the service asks for rather
than at one you picked.

In [ ]:
observation = SAMPLE_ARCHIVE if ROUTE is Route.SEND_SLIDES else SAMPLE_NPZ

job = client.predict(
    model=card["id"],
    observations={ROUTE.value: str(observation)},
)
print(job.id, job.state, "| queue position:", job.queue_position, "| eta:", job.eta_seconds)

In [ ]:
job = job.wait(on_poll=lambda j: print(f"  {j.state:<12} eta {j.eta_seconds}s", flush=True))
print("\nfinished:", job.state)
for event in job.events:
    print(" ", event)

In [ ]:
# `save` keeps the file. A result expires after a bounded time, so the file
# you downloaded is the copy of record.
job.save(PREDICTION)
print(PREDICTION, zenodo.human(PREDICTION.stat().st_size))

## 9. What came back

One `.h5ad`, the same layout whichever route produced it.

In [ ]:
import anndata

adata = anndata.read_h5ad(PREDICTION)
print(adata)

In [ ]:
layout = contracts.result_layout()
print("gene axis indexed by:", layout["var_index"])
print("coordinates are the tile's:", layout["spatial_anchor"])
print()
print("var columns:", list(adata.var.columns))
print(adata.var.head(3))
print()
print("obsm['spatial'] :", adata.obsm["spatial"].shape,
      "| x", adata.obsm["spatial"][:, 0].min(), "to", adata.obsm["spatial"][:, 0].max())

# Provenance: what produced this file, so it can be traced after the job has
# expired. `uns` also carries the full input specification; it is long, and
# nothing downstream needs it.
print()
print("model:", adata.uns["model"]["id"])
print("job:  ", adata.uns["job"]["id"])
print("scale:", adata.uns["resolution"])

### The one rule for reading it

**The gene axis is indexed by Ensembl stable gene identifier. The symbol
travels beside it, in `var["feature_name"]`.**

Symbols get renamed, retired and occasionally reassigned to a different gene.
An identifier does not. Indexing a result by symbol works until the day it
quietly returns the wrong column. Notebook 02 does the lookup properly, in the
open, and it is the first thing it does.

`var["measured_in_training"]` states per gene whether the model saw that gene
measured in training. A gene it never saw measured still has a column, and its
values read exactly like any other column in the file. That column is the one
place the file tells them apart.

In [ ]:
measured = adata.var["measured_in_training"].astype(bool)
print(f"{int(measured.sum()):,} of {adata.n_vars:,} genes were measured in training")
print(f"{int((~measured).sum()):,} were not")

## Next

[**02-downstream.ipynb**](02-downstream.ipynb) takes this file and draws a
named gene over the tissue, a UMAP of the predicted expression, and clusters
of spots put back on the slide.

| | Send your slides | Your slides stay with you |
|---|---|---|
| Runs on your machine | Tiling, quality control | Tiling, quality control, DeepSpot-H |
| Leaves your machine | Tile images, positions, quality measurements | Numbers, positions, quality measurements |
| Needs the `embed` extra | no | yes |
| Charged against | `tiles` | `rows` |
| Result | identical layout | identical layout |

[Privacy and retention](https://docs.auroraomics.org/privacy/) states what is
stored and for how long. Deleting a job removes its result, its record and its
copy of the inputs.